In [1]:
import matplotlib.pyplot as plt
import os
from sklearn.pipeline import Pipeline
from transformers import Resampler, BandPassFilter, Normalizer, MelSpectrogramTransformer
import cv2
import numpy as np
from matplotlib import cm

In [2]:
pipeline_melspec_img = Pipeline([
    ('resampler', Resampler()),
    ('bandpass', BandPassFilter()),
    ('normalizer', Normalizer(method='peak')),
    ('melspec', MelSpectrogramTransformer())
])

In [3]:
path = './audios_propios/ciclos_energy_based'
files = os.listdir(path)
paths = [os.path.join(path, f) for f in files]

In [4]:
X_propios_img = pipeline_melspec_img.fit_transform(paths)

In [6]:
def normalize_spec(spec):
    spec_min, spec_max = spec.min(), spec.max()
    spec = (spec - spec_min) / (spec_max - spec_min + 1e-6)
    spec = (spec * 255).astype(np.uint8)
    return spec

def spec_to_rgb(spec):
    spec = normalize_spec(spec)
    spec_color = cm.magma(spec / 255.0)[:, :, :3]  # RGB float 0–1
    return (spec_color * 255).astype(np.uint8)

def save_mel_images(X, files):
    output_dir = f'./audios_propios/ciclos_energy_based_mel_images/'
    os.makedirs(output_dir, exist_ok=True)

    filenames = []
    for i, spec in enumerate(X):
        spec_rgb = spec_to_rgb(np.flipud(spec))
        filename = os.path.splitext(files[i])[0] + ".png"
        cv2.imwrite(
            os.path.join(output_dir, filename),
            cv2.cvtColor(spec_rgb, cv2.COLOR_RGB2BGR)
        )
        filenames.append(filename)

save_mel_images(X_propios_img, files)